# 🎙️ Hybrid TTS: Edge-TTS + OpenVoice v2

Bu notebook Türkçe'de en yüksek kaliteyi elde etmek için iki aşamalı bir yöntem kullanır:
1. **Edge-TTS (Microsoft Azure):** Mükemmel Türkçe telaffuz ve duygu/hız (SSML) kontrolü ile temel sesi üretir.
2. **OpenVoice v2:** Üretilen temel sesin üzerine **sizin referans WAV dosyanızın** tınısını (ses rengini) klonlar.

⚠️ **Runtime -> Change runtime type -> GPU (T4)** seçili olduğundan emin olun.

## Hücre 1: Google Drive Bağlantısı

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/BITIRME"

ref_dir = os.path.join(PROJECT_DIR, "data/reference_voice")
if os.path.exists(ref_dir):
    print(f"✅ {len(os.listdir(ref_dir))} referans ses bulundu.")
else:
    print(f"❌ Referans klasörü bulunamadı: {ref_dir}")

## Hücre 2: Kurulumlar
Bu kütüphaneler Colab ile tam uyumludur ve çakışma yaratmaz.

In [ ]:
!pip install -q edge-tts openvoice-cli soundfile
print("✅ Kurulumlar tamamlandı!")

## Hücre 3: Model İndirme ve Yükleme

In [ ]:
import torch
import os
from openvoice.api import ToneColorConverter
from openvoice import se_extractor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"OpenVoice v2 Modeli Yükleniyor ({device})...")

# OpenVoice v2 checkpoint'ini indir
!wget -qN https://myshell-public-repo-host.s3.amazonaws.com/openvoice/checkpoints_v2_0417.zip
!unzip -qo checkpoints_v2_0417.zip

ckpt_converter = 'checkpoints_v2/converter'
tone_color_converter = ToneColorConverter(ckpt_converter, device=device)

# OpenVoice'in varsayılan kaynak ses embedding'ini yükle
source_se, audio_name = se_extractor.get_se('checkpoints_v2/base_speakers/ses/tr.wav', tone_color_converter, vad=False)

print("✅ OpenVoice v2 hazır!")

## Hücre 4: Test Cümleleri ve SSML Duygu Ayarları

In [ ]:
TEST_SENTENCES = {
    "neutral": "Merhaba, ben bankadan arıyorum. Hesabınızla ilgili bir bilgilendirme yapmak istiyorum.",
    "urgent": "Dikkat! Hesabınızda şüpheli bir işlem tespit ettik. Hemen doğrulama yapmanız gerekiyor.",
    "friendly": "İyi günler, size özel bir kampanyamız var. Birkaç dakikanızı alabilir miyim?",
    "worried": "Bir dakika, bu biraz garip geldi. Gerçekten bankadan mı arıyorsunuz?",
    "threatening": "Eğer beş dakika içinde bilgilerinizi doğrulamazsanız, hesabınız kalıcı olarak kapatılacak.",
}

EMOTION_SSML = {
    "neutral":     {"rate": "+0%",  "pitch": "+0%",  "volume": "+0%"},
    "friendly":    {"rate": "+5%",  "pitch": "+5%",  "volume": "+0%"},
    "worried":     {"rate": "-5%",  "pitch": "+10%", "volume": "-5%"},
    "urgent":      {"rate": "+15%", "pitch": "+5%",  "volume": "+10%"},
    "threatening": {"rate": "-10%", "pitch": "-10%", "volume": "+15%"},
}

AGENT_REF = os.path.join(PROJECT_DIR, "data/reference_voice/agent.wav")
VICTIM_REF = os.path.join(PROJECT_DIR, "data/reference_voice/victim.wav")

OUTPUT_DIR = "/content/hybrid_tts_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("✅ Hedef referans seslerinin (Target SE) embedding'leri çıkarılıyor...")
target_se_agent, _ = se_extractor.get_se(AGENT_REF, tone_color_converter, vad=True)
target_se_victim, _ = se_extractor.get_se(VICTIM_REF, tone_color_converter, vad=True)
print("✅ Embedding'ler çıkarıldı!")

## Hücre 5: Sentez ve Ses Klonlama İşlemi

In [ ]:
import asyncio
import edge_tts
import time
from IPython.display import Audio, display

async def generate_hybrid_audio():
    for emotion, text in TEST_SENTENCES.items():
        is_victim = (emotion == "worried")
        ref_se = target_se_victim if is_victim else target_se_agent
        # Microsoft'un Türkçe erkek (Ahmet) veya kadın (Emel) sesi
        voice = "tr-TR-EmelNeural" if is_victim else "tr-TR-AhmetNeural"
        ssml = EMOTION_SSML[emotion]
        
        edge_out = os.path.join(OUTPUT_DIR, f"{emotion}_1_edge.wav")
        final_out = os.path.join(OUTPUT_DIR, f"{emotion}_2_final.wav")

        print(f"\n🎭 [{emotion.upper()}] {text[:50]}...")
        start = time.time()
        
        # 1. Aşama: Edge-TTS ile duygu ve vurgu katılmış Türkçe sesi üret
        communicate = edge_tts.Communicate(
            text=text, 
            voice=voice,
            rate=ssml["rate"],
            pitch=ssml["pitch"],
            volume=ssml["volume"]
        )
        await communicate.save(edge_out)
        
        # 2. Aşama: OpenVoice ile tınıyı (ses rengini) klonla
        tone_color_converter.convert(
            audio_src_path=edge_out,
            src_se=source_se,
            tgt_se=ref_se,
            output_path=final_out,
            message="@MyShell"
        )
        elapsed = time.time() - start
        
        print(f"   ✅ {elapsed:.1f} saniyede tamamlandı.")
        print("   🔊 Aşama 1 (Edge-TTS Temel Ses - Duygulu Türkçe):")
        display(Audio(edge_out))
        print("   🔊 Aşama 2 (OpenVoice - Sizin Sesinize Klonlanmış):")
        display(Audio(final_out))

# Asenkron işlemi Colab'da çalıştır
await generate_hybrid_audio()

print("\n✅ İşlem tamamlandı! Ses dosyaları Drive'a kopyalanıyor...")

# Drive'a Kopyala
import shutil
save_dir = os.path.join(PROJECT_DIR, "outputs/hybrid_tts_test")
os.makedirs(save_dir, exist_ok=True)
for fname in os.listdir(OUTPUT_DIR):
    shutil.copy2(os.path.join(OUTPUT_DIR, fname), os.path.join(save_dir, fname))
print(f"📁 Dosyalar şuraya kaydedildi: {save_dir}")